## 26. تحلیل متن آگهی

حداقل عبارت‌های زیر بررسی شوند:

- نوساز
- کلیدنخورده
- فوری
- معاوضه
- زیر قیمت

مراحل مورد انتظار:

1. نرمال‌سازی متن فارسی
2. جست‌وجوی چند شکل نوشتاری
3. استخراج Featureهای باینری یا شمارشی
4. بررسی فراوانی
5. مقایسه قیمت خام
6. مقایسه کنترل‌شده برای واحدهای مشابه
7. بررسی Precision روی نمونه دستی

نتیجه باید مشخص کند که عبارت‌ها:

- با قیمت پایین‌تر یا بالاتر همراه‌اند؛
- یا پس از کنترل ویژگی‌ها رابطه قابل توجهی ندارند؛
- یا داده برای نتیجه‌گیری کافی نیست.

نرمال سازی متن فارسی در بخش تمیز کردن داده انجام شده است

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_feather("../Outputs/01_df.feather")

In [11]:
def extract_bi_ready_features(df: pd.DataFrame) -> pd.DataFrame:
    
    # 1. ستون‌های Boolean (برای فیلتر و محاسبات)
    keyword_columns = {
        'has_newly_built': ['نوساز', 'نوساخته', 'تازه‌ساز'],
        'has_never_lived': ['کلیدنخورده', 'کلید نخورده', 'صفر کلید'],
        'has_urgent': ['فوری', 'فورا', 'فوریت'],
        'has_swap': ['معاوضه', 'مبادله', 'تعویض'],
        'has_below_market': ['زیر قیمت', 'مناسب قیمت', 'قیمت عالی'],
        'has_discount': ['تخفیف', 'تخفیف ویژه'],
        # 'has_premium': ['ویژه', 'لوکس', 'لاکچری', 'مدرن', 'شیک'],
        # 'has_high_quality': ['فول', 'کامل', 'تمیز', 'مرتب', 'عالی']
    }
    
    # ترکیب متن
    df['search_text'] = df['title'].fillna('') + ' ' + df['description'].fillna('')
    
    # ایجاد ستون‌های Boolean
    for col_name, patterns in keyword_columns.items():
        pattern = '|'.join(patterns)
        df[col_name] = df['search_text'].str.contains(pattern, case=False, na=False)
    
    # 2. ستون Tag ترکیبی (برای نمایش)
    def create_tags(row):
        tags = []
        for col_name in keyword_columns.keys():
            if row[col_name]:
                tag = col_name.replace('has_', '')
                tags.append(tag)
        return '|'.join(tags) if tags else None
    
    df['keywords'] = df.apply(create_tags, axis=1)
    
    # 3. ستون شمارش
    df['keyword_count'] = df['keywords'].str.split('|').str.len()
    df['keyword_count'] = df['keyword_count'].fillna(0).astype(int)
    
    # 4. ستون اصلی‌ترین کلمه کلیدی (برای دسته‌بندی)
    def get_primary_keyword(row):
        if pd.isna(row['keywords']):
            return 'without_keyword'
        keywords = row['keywords'].split('|')
        # اولویت: نوساز > کلیدنخورده > معاوضه > زیر قیمت > فوری
        priority = ['newly_built', 'never_lived', 'swap', 'below_market', 'urgent']
        for p in priority:
            if p in keywords:
                return p
        return keywords[0] if keywords else 'without_keyword'
    
    df['primary_keyword'] = df.apply(get_primary_keyword, axis=1)
    
    # حذف ستون موقت
    df = df.drop('search_text', axis=1)
    
    return df


# اجرا روی دیتاست
df = extract_bi_ready_features(df)



In [12]:
has_columns = ["has_newly_built","has_never_lived","has_urgent","has_swap","has_below_market","has_discount"]


print("\n" + "=" * 50)
print(" جدول خلاصه:")
print("=" * 50)

summary_df = pd.DataFrame({
    'keyword': [col for col in has_columns],
    'count': [df[col].sum() for col in has_columns],
    'percent': [(df[col].sum() / len(df)) * 100 for col in has_columns]
})
summary_df = summary_df.sort_values('count', ascending=False)
print(summary_df.to_string(index=False))



 جدول خلاصه:
         keyword  count  percent
        has_swap  85833   8.5833
    has_discount  70725   7.0725
 has_newly_built  65018   6.5018
      has_urgent  43102   4.3102
 has_never_lived  39887   3.9887
has_below_market  37005   3.7005


In [ ]:
# # ستون‌های نهایی برای BI:
# bi_columns = {
#     'has_newly_built': 'آگهی نوساز',
#     'has_never_lived': 'آگهی کلیدنخورده', 
#     'has_urgent': 'آگهی فوری',
#     'has_swap': 'آگهی معاوضه',
#     'has_below_market': 'آگهی زیر قیمت',
#     'has_discount': 'آگهی تخفیف‌دار',
#     'has_premium': 'آگهی ویژه/لوکس',
#     'has_high_quality': 'آگهی با کیفیت بالا',
#     'keywords': 'لیست کلمات کلیدی (با | جدا شده)',
#     'keyword_count': 'تعداد کلمات کلیدی',
#     'primary_keyword': 'کلمه کلیدی اصلی'
# }

# print("✅ ستون‌های آماده برای BI:")
# for col, desc in bi_columns.items():
#     if col in df.columns:
#         print(f"   {col}: {desc}")

In [13]:
df.to_feather("../Outputs/26_df.feather")